# Segmenter: official ViT encoder and Mask Transformer on BTXRD

Fine-tune the original Segmenter implementation on binary BTXRD masks for 50 epochs. The upstream ADE20K segmentation checkpoint link is unavailable; ViT AugReg pretrained weights initialize the encoder and the two-class decoder starts from scratch.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

assert Path('/kaggle/working').is_dir(), 'This notebook must run on Kaggle.'
os.environ['HF_HOME'] = '/kaggle/temp/huggingface'
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.31', 'einops==0.8.1'], check=True)
import timm
import torch
from huggingface_hub import hf_hub_download

assert torch.cuda.is_available(), 'CUDA GPU unavailable: check the Kaggle accelerator and GPU quota.'
print('Torch:', torch.__version__, 'timm:', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0), flush=True)


In [ ]:
import hashlib

DATASET_REVISION = 'cf8b0569b18e43e7cecab412a06e60aeaf258b32'
EXPECTED_SHA256 = '59ce94244fdf997f0c68e6a378690a4359e6c411f7fda7ac2f3994489637884b'
archive = Path(hf_hub_download(repo_id='torusama/xla-seminar-uia', repo_type='dataset',
                               filename='btxrd_final.zip', revision=DATASET_REVISION))
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
if digest != EXPECTED_SHA256:
    raise RuntimeError(f'Dataset checksum mismatch: {digest}')
print('BTXRD downloaded; SHA-256:', digest, flush=True)


In [ ]:
# Official Segmenter source is bundled with its MIT license.
PROJECT = Path('/kaggle/temp/xla-segmenter-official')
SOURCES = {'src/xla/__init__.py': '"""Research scaffold for vision classification and segmentation."""\n\n__version__ = "0.1.0"\n', 'src/xla/data/__init__.py': '"""Data interfaces for future phases."""\n', 'src/xla/data/hf_segmentation.py': '"""Read the pinned Hugging Face archive and rasterize polygon masks."""\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch.utils.data import Dataset\nfrom torchvision.transforms import functional as TF\n\n\nclass PolygonArchive(Dataset):\n    """Binary union of all annotated polygons; normal samples have empty masks."""\n\n    def __init__(self, archive: Path, split: str, excluded_hashes: set[str] | None = None):\n        self.archive = Path(archive)\n        self.samples = []\n        self.excluded = 0\n        self._zip = None\n        self._zip_pid = None\n        excluded_hashes = excluded_hashes or set()\n        with ZipFile(self.archive) as z:\n            names = set(z.namelist())\n            annotations = sorted(n for n in names if n.startswith(f"{split}/annotations/")\n                                 and n.endswith(".json"))\n            for name in annotations:\n                annotation = json.loads(z.read(name))\n                category = Path(name).parent.name\n                image_name = f"{split}/images/{category}/{annotation[\'image_name\']}"\n                if image_name not in names:\n                    raise ValueError(f"Missing image for {name}: {image_name}")\n                digest = hashlib.sha256(z.read(image_name)).hexdigest()\n                if digest in excluded_hashes:\n                    self.excluded += 1\n                    continue\n                self.samples.append((image_name, name, digest))\n        if not self.samples:\n            raise ValueError(f"No samples in {self.archive} split {split}")\n\n    @property\n    def hashes(self) -> set[str]:\n        return {sample[2] for sample in self.samples}\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getstate__(self):\n        state = self.__dict__.copy()\n        state["_zip"] = None\n        state["_zip_pid"] = None\n        return state\n\n    def __getitem__(self, index):\n        image_name, annotation_name, _ = self.samples[index]\n        if self._zip is None or self._zip_pid != os.getpid():\n            if self._zip is not None:\n                self._zip.close()\n            self._zip = ZipFile(self.archive)\n            self._zip_pid = os.getpid()\n        annotation = json.loads(self._zip.read(annotation_name))\n        with Image.open(io.BytesIO(self._zip.read(image_name))) as source:\n            image = source.convert("RGB")\n        expected = (annotation["image_width"], annotation["image_height"])\n        if image.size != expected or expected != (224, 224):\n            raise ValueError(f"Image and annotation dimensions do not match 224x224: {image_name}")\n        mask = Image.new("L", image.size, 0)\n        draw = ImageDraw.Draw(mask)\n        count = 0\n        for item in annotation["annotations"]:\n            for polygon in item.get("polygons", []):\n                if len(polygon) < 3:\n                    raise ValueError(f"Invalid polygon: {annotation_name}")\n                draw.polygon([tuple(point) for point in polygon], fill=1)\n                count += 1\n        if not count and annotation["class_label"].lower() != "normal":\n            raise ValueError(f"Non-normal sample has no polygons: {annotation_name}")\n        pixels = TF.normalize(TF.to_tensor(image), mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))\n        target = torch.frombuffer(bytearray(mask.tobytes()), dtype=torch.uint8).reshape(1, 224, 224)\n        return pixels, target.float()\n\n\ndef make_splits(archive: Path):\n    """Remove conflicting labels and train duplicates of held-out images."""\n    splits = [PolygonArchive(archive, name) for name in ("train", "val", "test")]\n    categories = {}\n    for split in splits:\n        for image_name, _, digest in split.samples:\n            categories.setdefault(digest, set()).add(Path(image_name).parent.name)\n    conflicts = {digest for digest, labels in categories.items() if len(labels) > 1}\n    for split in splits:\n        original = len(split.samples)\n        split.samples = [sample for sample in split.samples if sample[2] not in conflicts]\n        split.excluded = original - len(split.samples)\n    train, val, test = splits\n    if val.hashes & test.hashes:\n        raise ValueError("Validation and test contain identical images; resolve the split first")\n    held_out = val.hashes | test.hashes\n    original = len(train.samples)\n    train.samples = [sample for sample in train.samples if sample[2] not in held_out]\n    train.excluded += original - len(train.samples)\n    if any(not split.samples for split in splits):\n        raise ValueError("A split is empty after removing conflicting and duplicate images")\n    return train, val, test\n', 'src/xla/engine/__init__.py': '"""Engine interfaces for future phases."""\n', 'src/xla/engine/segmenter_official.py': '"""Per-pixel cross-entropy and measurements for the upstream Segmenter."""\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.data import Dataset\n\n\nclass RandomHorizontalFlip(Dataset):\n    """Apply the same horizontal flip to images and binary masks."""\n\n    def __init__(self, dataset, probability=0.5):\n        self.dataset = dataset\n        self.probability = probability\n\n    def __len__(self):\n        return len(self.dataset)\n\n    def __getitem__(self, index):\n        image, mask = self.dataset[index]\n        if torch.rand(()) < self.probability:\n            image, mask = image.flip(-1), mask.flip(-1)\n        return image, mask\n\n\ndef run_epoch(model, loader, device, optimizer=None, scheduler=None):\n    training = optimizer is not None\n    model.train(training)\n    total_loss = count = tp = fp = fn = 0\n    with torch.set_grad_enabled(training):\n        for images, target in loader:\n            images, target = images.to(device), target.to(device)\n            if training:\n                optimizer.zero_grad()\n            scores = model(images)\n            loss = F.cross_entropy(scores, target[:, 0].long(), ignore_index=255)\n            if training:\n                loss.backward()\n                optimizer.step()\n                if scheduler is not None:\n                    scheduler.step()\n            count += images.shape[0]\n            total_loss += loss.item() * images.shape[0]\n            prediction = scores.detach().argmax(dim=1).bool()\n            truth = target[:, 0].bool()\n            tp += (prediction & truth).sum().item()\n            fp += (prediction & ~truth).sum().item()\n            fn += (~prediction & truth).sum().item()\n    return {"loss": total_loss / count,\n            "dice": 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 1.0,\n            "iou": tp / (tp + fp + fn) if tp + fp + fn else 1.0}\n', 'src/xla/models/__init__.py': '"""Models interfaces for future phases."""\n', 'src/xla/models/segmenter_official.py': '"""Segmenter built from the authors\' ViT and Mask Transformer modules."""\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom xla.vendor.segmenter.decoder import MaskTransformer\nfrom xla.vendor.segmenter.vit import VisionTransformer\n\nBACKBONE = "vit_base_patch16_384.augreg_in21k_ft_in1k"\n\n\nclass OfficialSegmenter(nn.Module):\n    """Two-class Segmenter with the upstream encoder and decoder."""\n\n    def __init__(self, pretrained=True, image_size=224):\n        super().__init__()\n        self.encoder = VisionTransformer(\n            image_size=(image_size, image_size), patch_size=16,\n            n_layers=12, d_model=768, d_ff=3072, n_heads=12,\n            n_cls=1000, dropout=0.0, drop_path_rate=0.1,\n        )\n        if pretrained:\n            import timm\n\n            source = timm.create_model(BACKBONE, pretrained=True, img_size=image_size)\n            self.encoder.load_state_dict(source.state_dict(), strict=True)\n            del source\n        self.decoder = MaskTransformer(\n            n_cls=2, patch_size=16, d_encoder=768, n_layers=2,\n            n_heads=12, d_model=768, d_ff=3072, drop_path_rate=0.0, dropout=0.1,\n        )\n\n    def forward(self, images):\n        tokens = self.encoder(images, return_features=True)[:, 1:]\n        masks = self.decoder(tokens, images.shape[-2:])\n        return F.interpolate(masks, size=images.shape[-2:],\n                             mode="bilinear", align_corners=False)\n', 'src/xla/vendor/__init__.py': '"""Vendored third-party components."""\n', 'src/xla/vendor/segmenter/__init__.py': '"""Segmenter decoder; see LICENSE and docs/vit/segmenter_source.md."""\n', 'src/xla/vendor/segmenter/blocks.py': '"""\nAdapted from 2020 Ross Wightman\nhttps://github.com/rwightman/pytorch-image-models\n"""\n\nimport torch\nimport torch.nn as nn\nfrom einops import rearrange\nfrom pathlib import Path\n\nimport torch.nn.functional as F\n\nfrom timm.layers import DropPath\n\n\nclass FeedForward(nn.Module):\n    def __init__(self, dim, hidden_dim, dropout, out_dim=None):\n        super().__init__()\n        self.fc1 = nn.Linear(dim, hidden_dim)\n        self.act = nn.GELU()\n        if out_dim is None:\n            out_dim = dim\n        self.fc2 = nn.Linear(hidden_dim, out_dim)\n        self.drop = nn.Dropout(dropout)\n\n    @property\n    def unwrapped(self):\n        return self\n\n    def forward(self, x):\n        x = self.fc1(x)\n        x = self.act(x)\n        x = self.drop(x)\n        x = self.fc2(x)\n        x = self.drop(x)\n        return x\n\n\nclass Attention(nn.Module):\n    def __init__(self, dim, heads, dropout):\n        super().__init__()\n        self.heads = heads\n        head_dim = dim // heads\n        self.scale = head_dim ** -0.5\n        self.attn = None\n\n        self.qkv = nn.Linear(dim, dim * 3)\n        self.attn_drop = nn.Dropout(dropout)\n        self.proj = nn.Linear(dim, dim)\n        self.proj_drop = nn.Dropout(dropout)\n\n    @property\n    def unwrapped(self):\n        return self\n\n    def forward(self, x, mask=None):\n        B, N, C = x.shape\n        qkv = (\n            self.qkv(x)\n            .reshape(B, N, 3, self.heads, C // self.heads)\n            .permute(2, 0, 3, 1, 4)\n        )\n        q, k, v = (\n            qkv[0],\n            qkv[1],\n            qkv[2],\n        )\n\n        attn = (q @ k.transpose(-2, -1)) * self.scale\n        attn = attn.softmax(dim=-1)\n        attn = self.attn_drop(attn)\n\n        x = (attn @ v).transpose(1, 2).reshape(B, N, C)\n        x = self.proj(x)\n        x = self.proj_drop(x)\n\n        return x, attn\n\n\nclass Block(nn.Module):\n    def __init__(self, dim, heads, mlp_dim, dropout, drop_path):\n        super().__init__()\n        self.norm1 = nn.LayerNorm(dim)\n        self.norm2 = nn.LayerNorm(dim)\n        self.attn = Attention(dim, heads, dropout)\n        self.mlp = FeedForward(dim, mlp_dim, dropout)\n        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()\n\n    def forward(self, x, mask=None, return_attention=False):\n        y, attn = self.attn(self.norm1(x), mask)\n        if return_attention:\n            return attn\n        x = x + self.drop_path(y)\n        x = x + self.drop_path(self.mlp(self.norm2(x)))\n        return x\n', 'src/xla/vendor/segmenter/decoder.py': 'import math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom einops import rearrange\n\nfrom timm.layers import trunc_normal_\n\nfrom xla.vendor.segmenter.blocks import Block\nfrom xla.vendor.segmenter.utils import init_weights\n\n\nclass DecoderLinear(nn.Module):\n    def __init__(self, n_cls, patch_size, d_encoder):\n        super().__init__()\n\n        self.d_encoder = d_encoder\n        self.patch_size = patch_size\n        self.n_cls = n_cls\n\n        self.head = nn.Linear(self.d_encoder, n_cls)\n        self.apply(init_weights)\n\n    @torch.jit.ignore\n    def no_weight_decay(self):\n        return set()\n\n    def forward(self, x, im_size):\n        H, W = im_size\n        GS = H // self.patch_size\n        x = self.head(x)\n        x = rearrange(x, "b (h w) c -> b c h w", h=GS)\n\n        return x\n\n\nclass MaskTransformer(nn.Module):\n    def __init__(\n        self,\n        n_cls,\n        patch_size,\n        d_encoder,\n        n_layers,\n        n_heads,\n        d_model,\n        d_ff,\n        drop_path_rate,\n        dropout,\n    ):\n        super().__init__()\n        self.d_encoder = d_encoder\n        self.patch_size = patch_size\n        self.n_layers = n_layers\n        self.n_cls = n_cls\n        self.d_model = d_model\n        self.d_ff = d_ff\n        self.scale = d_model ** -0.5\n\n        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, n_layers)]\n        self.blocks = nn.ModuleList(\n            [Block(d_model, n_heads, d_ff, dropout, dpr[i]) for i in range(n_layers)]\n        )\n\n        self.cls_emb = nn.Parameter(torch.randn(1, n_cls, d_model))\n        self.proj_dec = nn.Linear(d_encoder, d_model)\n\n        self.proj_patch = nn.Parameter(self.scale * torch.randn(d_model, d_model))\n        self.proj_classes = nn.Parameter(self.scale * torch.randn(d_model, d_model))\n\n        self.decoder_norm = nn.LayerNorm(d_model)\n        self.mask_norm = nn.LayerNorm(n_cls)\n\n        self.apply(init_weights)\n        trunc_normal_(self.cls_emb, std=0.02)\n\n    @torch.jit.ignore\n    def no_weight_decay(self):\n        return {"cls_emb"}\n\n    def forward(self, x, im_size):\n        H, W = im_size\n        GS = H // self.patch_size\n\n        x = self.proj_dec(x)\n        cls_emb = self.cls_emb.expand(x.size(0), -1, -1)\n        x = torch.cat((x, cls_emb), 1)\n        for blk in self.blocks:\n            x = blk(x)\n        x = self.decoder_norm(x)\n\n        patches, cls_seg_feat = x[:, : -self.n_cls], x[:, -self.n_cls :]\n        patches = patches @ self.proj_patch\n        cls_seg_feat = cls_seg_feat @ self.proj_classes\n\n        patches = patches / patches.norm(dim=-1, keepdim=True)\n        cls_seg_feat = cls_seg_feat / cls_seg_feat.norm(dim=-1, keepdim=True)\n\n        masks = patches @ cls_seg_feat.transpose(1, 2)\n        masks = self.mask_norm(masks)\n        masks = rearrange(masks, "b (h w) n -> b n h w", h=int(GS))\n\n        return masks\n\n    def get_attention_map(self, x, layer_id):\n        if layer_id >= self.n_layers or layer_id < 0:\n            raise ValueError(\n                f"Provided layer_id: {layer_id} is not valid. 0 <= {layer_id} < {self.n_layers}."\n            )\n        x = self.proj_dec(x)\n        cls_emb = self.cls_emb.expand(x.size(0), -1, -1)\n        x = torch.cat((x, cls_emb), 1)\n        for i, blk in enumerate(self.blocks):\n            if i < layer_id:\n                x = blk(x)\n            else:\n                return blk(x, return_attention=True)\n', 'src/xla/vendor/segmenter/utils.py': '"""Initialization helper from Segmenter; see LICENSE."""\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom timm.layers import trunc_normal_\n\n\ndef init_weights(m):\n    if isinstance(m, nn.Linear):\n        trunc_normal_(m.weight, std=0.02)\n        if isinstance(m, nn.Linear) and m.bias is not None:\n            nn.init.constant_(m.bias, 0)\n    elif isinstance(m, nn.LayerNorm):\n        nn.init.constant_(m.bias, 0)\n        nn.init.constant_(m.weight, 1.0)\n\ndef resize_pos_embed(posemb, grid_old_shape, grid_new_shape, num_extra_tokens):\n    # Rescale the grid of position embeddings when loading from state_dict. Adapted from\n    # https://github.com/google-research/vision_transformer/blob/00883dd691c63a6830751563748663526e811cee/vit_jax/checkpoint.py#L224\n    posemb_tok, posemb_grid = (\n        posemb[:, :num_extra_tokens],\n        posemb[0, num_extra_tokens:],\n    )\n    if grid_old_shape is None:\n        gs_old_h = int(math.sqrt(len(posemb_grid)))\n        gs_old_w = gs_old_h\n    else:\n        gs_old_h, gs_old_w = grid_old_shape\n\n    gs_h, gs_w = grid_new_shape\n    posemb_grid = posemb_grid.reshape(1, gs_old_h, gs_old_w, -1).permute(0, 3, 1, 2)\n    posemb_grid = F.interpolate(posemb_grid, size=(gs_h, gs_w), mode="bilinear")\n    posemb_grid = posemb_grid.permute(0, 2, 3, 1).reshape(1, gs_h * gs_w, -1)\n    posemb = torch.cat([posemb_tok, posemb_grid], dim=1)\n    return posemb\n\n\n', 'src/xla/vendor/segmenter/vit.py': '"""\nAdapted from 2020 Ross Wightman\nhttps://github.com/rwightman/pytorch-image-models\n"""\n\nimport torch\nimport torch.nn as nn\n\nfrom xla.vendor.segmenter.utils import init_weights, resize_pos_embed\nfrom xla.vendor.segmenter.blocks import Block\n\nfrom timm.layers import DropPath\nfrom timm.layers import trunc_normal_\nfrom timm.models.vision_transformer import _load_weights\n\n\nclass PatchEmbedding(nn.Module):\n    def __init__(self, image_size, patch_size, embed_dim, channels):\n        super().__init__()\n\n        self.image_size = image_size\n        if image_size[0] % patch_size != 0 or image_size[1] % patch_size != 0:\n            raise ValueError("image dimensions must be divisible by the patch size")\n        self.grid_size = image_size[0] // patch_size, image_size[1] // patch_size\n        self.num_patches = self.grid_size[0] * self.grid_size[1]\n        self.patch_size = patch_size\n\n        self.proj = nn.Conv2d(\n            channels, embed_dim, kernel_size=patch_size, stride=patch_size\n        )\n\n    def forward(self, im):\n        B, C, H, W = im.shape\n        x = self.proj(im).flatten(2).transpose(1, 2)\n        return x\n\n\nclass VisionTransformer(nn.Module):\n    def __init__(\n        self,\n        image_size,\n        patch_size,\n        n_layers,\n        d_model,\n        d_ff,\n        n_heads,\n        n_cls,\n        dropout=0.1,\n        drop_path_rate=0.0,\n        distilled=False,\n        channels=3,\n    ):\n        super().__init__()\n        self.patch_embed = PatchEmbedding(\n            image_size,\n            patch_size,\n            d_model,\n            channels,\n        )\n        self.patch_size = patch_size\n        self.n_layers = n_layers\n        self.d_model = d_model\n        self.d_ff = d_ff\n        self.n_heads = n_heads\n        self.dropout = nn.Dropout(dropout)\n        self.n_cls = n_cls\n\n        # cls and pos tokens\n        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))\n        self.distilled = distilled\n        if self.distilled:\n            self.dist_token = nn.Parameter(torch.zeros(1, 1, d_model))\n            self.pos_embed = nn.Parameter(\n                torch.randn(1, self.patch_embed.num_patches + 2, d_model)\n            )\n            self.head_dist = nn.Linear(d_model, n_cls)\n        else:\n            self.pos_embed = nn.Parameter(\n                torch.randn(1, self.patch_embed.num_patches + 1, d_model)\n            )\n\n        # transformer blocks\n        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, n_layers)]\n        self.blocks = nn.ModuleList(\n            [Block(d_model, n_heads, d_ff, dropout, dpr[i]) for i in range(n_layers)]\n        )\n\n        # output head\n        self.norm = nn.LayerNorm(d_model)\n        self.head = nn.Linear(d_model, n_cls)\n\n        trunc_normal_(self.pos_embed, std=0.02)\n        trunc_normal_(self.cls_token, std=0.02)\n        if self.distilled:\n            trunc_normal_(self.dist_token, std=0.02)\n        self.pre_logits = nn.Identity()\n\n        self.apply(init_weights)\n\n    @torch.jit.ignore\n    def no_weight_decay(self):\n        return {"pos_embed", "cls_token", "dist_token"}\n\n    @torch.jit.ignore()\n    def load_pretrained(self, checkpoint_path, prefix=""):\n        _load_weights(self, checkpoint_path, prefix)\n\n    def forward(self, im, return_features=False):\n        B, _, H, W = im.shape\n        PS = self.patch_size\n\n        x = self.patch_embed(im)\n        cls_tokens = self.cls_token.expand(B, -1, -1)\n        if self.distilled:\n            dist_tokens = self.dist_token.expand(B, -1, -1)\n            x = torch.cat((cls_tokens, dist_tokens, x), dim=1)\n        else:\n            x = torch.cat((cls_tokens, x), dim=1)\n\n        pos_embed = self.pos_embed\n        num_extra_tokens = 1 + self.distilled\n        if x.shape[1] != pos_embed.shape[1]:\n            pos_embed = resize_pos_embed(\n                pos_embed,\n                self.patch_embed.grid_size,\n                (H // PS, W // PS),\n                num_extra_tokens,\n            )\n        x = x + pos_embed\n        x = self.dropout(x)\n\n        for blk in self.blocks:\n            x = blk(x)\n        x = self.norm(x)\n\n        if return_features:\n            return x\n\n        if self.distilled:\n            x, x_dist = x[:, 0], x[:, 1]\n            x = self.head(x)\n            x_dist = self.head_dist(x_dist)\n            x = (x + x_dist) / 2\n        else:\n            x = x[:, 0]\n            x = self.head(x)\n        return x\n\n    def get_attention_map(self, im, layer_id):\n        if layer_id >= self.n_layers or layer_id < 0:\n            raise ValueError(\n                f"Provided layer_id: {layer_id} is not valid. 0 <= {layer_id} < {self.n_layers}."\n            )\n        B, _, H, W = im.shape\n        PS = self.patch_size\n\n        x = self.patch_embed(im)\n        cls_tokens = self.cls_token.expand(B, -1, -1)\n        if self.distilled:\n            dist_tokens = self.dist_token.expand(B, -1, -1)\n            x = torch.cat((cls_tokens, dist_tokens, x), dim=1)\n        else:\n            x = torch.cat((cls_tokens, x), dim=1)\n\n        pos_embed = self.pos_embed\n        num_extra_tokens = 1 + self.distilled\n        if x.shape[1] != pos_embed.shape[1]:\n            pos_embed = resize_pos_embed(\n                pos_embed,\n                self.patch_embed.grid_size,\n                (H // PS, W // PS),\n                num_extra_tokens,\n            )\n        x = x + pos_embed\n\n        for i, blk in enumerate(self.blocks):\n            if i < layer_id:\n                x = blk(x)\n            else:\n                return blk(x, return_attention=True)\n', 'src/xla/vendor/segmenter/LICENSE': 'MIT License\n\nCopyright (c) 2021 Robin Strudel\nCopyright (c) INRIA\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'scripts/train_segmenter_official.py': '"""Fine-tune the authors\' Segmenter architecture and training method on BTXRD."""\nimport argparse\nimport hashlib\nimport json\nimport random\nfrom pathlib import Path\n\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom xla.data.hf_segmentation import make_splits\nfrom xla.engine.segmenter_official import RandomHorizontalFlip, run_epoch\nfrom xla.models.segmenter_official import BACKBONE, OfficialSegmenter\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--archive", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--epochs", type=int, default=50)\n    parser.add_argument("--batch-size", type=int, default=4)\n    parser.add_argument("--workers", type=int, default=0)\n    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--from-scratch", action="store_true")\n    args = parser.parse_args()\n    if args.epochs < 1 or args.batch_size < 1 or args.workers < 0:\n        parser.error("epochs and batch-size must be positive; workers must be nonnegative")\n    if args.output.exists():\n        raise FileExistsError(f"Output directory already exists: {args.output}")\n    device = torch.device("cuda" if args.device == "auto" and torch.cuda.is_available()\n                          else "cpu" if args.device == "auto" else args.device)\n    if device.type == "cuda" and not torch.cuda.is_available():\n        parser.error("CUDA is not available")\n    random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(args.seed)\n    train_set, val_set, test_set = make_splits(args.archive)\n    print(f"Samples: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}", flush=True)\n    train_loader = DataLoader(RandomHorizontalFlip(train_set), batch_size=args.batch_size,\n                              shuffle=True, num_workers=args.workers)\n    val_loader = DataLoader(val_set, batch_size=args.batch_size, num_workers=args.workers)\n    model = OfficialSegmenter(pretrained=not args.from_scratch).to(device)\n    base_lr, min_lr, power = 0.001, 0.00001, 0.9\n    optimizer = torch.optim.SGD(model.parameters(), lr=base_lr, momentum=0.9, weight_decay=0)\n    max_steps = args.epochs * len(train_loader)\n    scheduler = torch.optim.lr_scheduler.LambdaLR(\n        optimizer, lambda step: (min_lr + (base_lr - min_lr) *\n                                 max(0.0, 1 - step / max_steps) ** power) / base_lr)\n    args.output.mkdir(parents=True, exist_ok=False)\n    settings = {"architecture": "segmenter_official_mask_transformer",\n                "source_commit": "20d1bfad354165ee45c3f65972a4d9c131f58d53",\n                "backbone": BACKBONE, "pretrained": not args.from_scratch,\n                "pretrained_decoder": False, "image_size": 224, "classes": 2,\n                "loss": "cross_entropy", "optimizer": "SGD", "momentum": 0.9,\n                "weight_decay": 0.0, "learning_rate": base_lr, "min_lr": min_lr,\n                "scheduler": "polynomial", "poly_power": power,\n                "epochs": args.epochs, "batch_size": args.batch_size, "seed": args.seed,\n                "augmentation": "horizontal_flip", "archive_sha256": sha256(args.archive),\n                "excluded_train": train_set.excluded, "excluded_val": val_set.excluded,\n                "excluded_test": test_set.excluded}\n    (args.output / "config.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")\n    print(f"Starting official Segmenter training: {args.epochs} epochs, device={device}", flush=True)\n    history, best = [], -1.0\n    for epoch in range(1, args.epochs + 1):\n        training = run_epoch(model, train_loader, device, optimizer, scheduler)\n        validation = run_epoch(model, val_loader, device)\n        record = {"epoch": epoch, "train": training, "val": validation,\n                  "learning_rate": optimizer.param_groups[0]["lr"]}\n        history.append(record)\n        print(json.dumps(record), flush=True)\n        (args.output / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")\n        checkpoint = {"model": model.state_dict(), "epoch": epoch, "val": validation,\n                      "archive_sha256": settings["archive_sha256"], "settings": settings}\n        if validation["dice"] > best:\n            best = validation["dice"]\n            torch.save(checkpoint, args.output / "best.pt")\n        if epoch == args.epochs:\n            torch.save(checkpoint, args.output / "final.pt")\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, content in SOURCES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
os.environ['PYTHONPATH'] = str(PROJECT / 'src')
print('Project sources prepared:', len(SOURCES), 'files', flush=True)


In [ ]:
# 50 epochs; best checkpoint is selected using validation Dice.
output = Path('/kaggle/working/segmenter_official_btxrd_50e')
command = [sys.executable, '-u', str(PROJECT / 'scripts/train_segmenter_official.py'),
           '--archive', str(archive), '--output', str(output), '--epochs', '50',
           '--batch-size', '4', '--workers', '2', '--seed', '42', '--device', 'cuda']
subprocess.run(command, check=True, env=os.environ.copy())
assert (output / 'best.pt').is_file() and (output / 'final.pt').is_file()
print('Finished 50 epochs. Output files:', [p.name for p in output.iterdir()], flush=True)


Output contains best.pt (validation Dice), final.pt, config.json and history.json. The test split is not used for model selection.
